### Iris Dataset with NN

In [10]:
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [11]:
# Loading the dataset
iris = load_iris()
X = iris.data
y = iris.target

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
X.dtype, y.dtype

(dtype('float64'), dtype('int64'))

In [14]:
# Since all features are numbers, standardize
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score

# print(f"Before:\nTrain: {X_train[:2]}\nTest:{X_test[:2]}")

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# print(f"After:\nTrain: {X_train[:2]}\nTest:{X_test[:2]}")

In [15]:
# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)  # Labels so need integers
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [16]:
# Iris Classification Model Class
class IrisNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_one  = nn.Linear(4, 10)
        self.relu = nn.ReLU()
        self.layer_two = nn.Linear(10, 3)

    def forward(self, x):
        # return self.layer_one(x)
        return self.layer_two(self.relu(self.layer_one(x)))

In [17]:
def train_model(model, X_train, y_train, loss_fn, optimizer, epochs=100):
    for epoch in range(epochs):
        y_pred = model(X_train)
        loss = loss_fn(y_pred, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")
    return model

def evaluate_model(model, X_test, y_test):
    with torch.no_grad():
        y_pred = model(X_test)
        _, predicted = torch.max(y_pred, 1)
        accuracy = accuracy_score(y_test, predicted)
        print(f"\nTest Accuracy: {accuracy * 100:.3f}%")
        return f"Predictions: {predicted[:5]} True: {y_test[:5]}"

In [18]:
torch.manual_seed(42)
ModelV2 = IrisNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(ModelV2.parameters(), lr=0.01)

train_model(ModelV2, X_train_tensor, y_train_tensor, loss_fn, optimizer)
evaluate_model(ModelV2, X_test_tensor, y_test_tensor)

Epoch 10/100, Loss: 0.9405
Epoch 20/100, Loss: 0.7974
Epoch 30/100, Loss: 0.6491
Epoch 40/100, Loss: 0.5285
Epoch 50/100, Loss: 0.4448
Epoch 60/100, Loss: 0.3831
Epoch 70/100, Loss: 0.3333
Epoch 80/100, Loss: 0.2910
Epoch 90/100, Loss: 0.2550
Epoch 100/100, Loss: 0.2244

Test Accuracy: 96.667%


'Predictions: tensor([1, 0, 2, 1, 1]) True: tensor([1, 0, 2, 1, 1])'

### Batch - Iteration - Epoch
- **Batch**: Small group of training examples processed together in one forward/backward pass
    - Break dataset into batches and feed, saves memory and speeds up training

- **Iteration**: One update to model's weight using one batch
    - 1 Iteration = Processing 1 Batch

- **Epoch**: Model has seen the entire training dataset once
    - Data split into batches, one epoch has multiple iterations(one iteration per batch)

Example:
- Training Dataset has 1000 images

    - Batch Size = 10
    
    - Total Batches = 1000 / 10 = 10 Batches
    
    - Each batch has 100 images
    
    - One iteration means processing 1 batch or 100 images
    
    - One epoch means model has processed all 10 batches or 1000 images once
    
    - Training for 5 epochs, model will see the entire dataset 5 times (50 iterations in total)